# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ruzaki11/Flyrank-ml-intern-tasks/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I will begin with Logistic Regression because the task is binary classification and Logistic Regression provides a simple, interpretable ML benchmark. I will then consider Random Forest as a more flexible comparison model to capture nonlinear relationships and interactions. The models will be judged against the Week-4 rule-based baseline using the same test data and evaluation metric.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [19]:
import pandas as pd

df = pd.read_csv("hf://datasets/FlyRank/internship-starter/content_refresh_anonymized.csv")
print(df.shape)

(30000, 53)


In [20]:
label = "is_initial_refresh_candidate"



In [21]:
y = df[label].astype(bool).astype(int)

print(y.dtype)
print(y.unique())
print(y.value_counts())

int64
[1 0]
is_initial_refresh_candidate
0    18608
1    11392
Name: count, dtype: int64


In [22]:
from sklearn.model_selection import StratifiedGroupKFold

groups = df["client_id"]

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

train_idx, test_idx = next(
    cv.split(
        df,
        y,
        groups=groups
    )
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

Client grouping prevents same-client information from appearing in both train and test; a reliable time split isn't supported by the available timestamps.

I did not use a time-aware split because the dataset does not provide a clear prediction/observation timestamp that would allow a reliable chronological train-versus-future-test split.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [23]:
excluded = [
    "content_id",
    "client_id",
    "is_initial_refresh_candidate",

    # W03 leakage / label-derived fields
    "needs_indexing",
    "is_quick_win",
    "needs_ctr_fix",
    "needs_engagement_fix",
    "is_underperformer",
    "is_declining",
    "health_score",
    "ai_opportunity",

    # future-window fields, if present
    "future_clicks",
    "future_ctr",
    "next_30d_sessions",
]

In [24]:
existing_excluded = [
    col for col in excluded
    if col in df.columns
]

print(existing_excluded)

['content_id', 'client_id', 'is_initial_refresh_candidate', 'needs_indexing', 'is_quick_win', 'needs_ctr_fix', 'needs_engagement_fix', 'is_underperformer', 'is_declining', 'health_score', 'ai_opportunity']


In [25]:
feature_columns = [
    col for col in df.columns
    if col not in existing_excluded
]

print("Number of candidate features:", len(feature_columns))
print(feature_columns)

Number of candidate features: 42
['search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [26]:
# separate the categorical features from the numeric features

numeric_features = df[feature_columns].select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = df[feature_columns].select_dtypes(
    include=["object", "bool"]
).columns.tolist()

print("Numeric:", len(numeric_features))
print(numeric_features)

print("\nCategorical:", len(categorical_features))
print(categorical_features)

Numeric: 30
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'trend_pct']

Categorical: 12
['competition_level', 'content_type', 'main_intent', 'provider_used', 'model_used', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier', 'trend_direction']


In [27]:
X_train = train_df[feature_columns]
X_test = test_df[feature_columns]

In [28]:
# the processor pipeline

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])


In [29]:
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("preprocessor" , preprocessor),
    ("classifier" , LogisticRegression(
        random_state=42,
        max_iter=1000
    ))
])

model.fit(X_train , y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['search_volume',
                                                   'competition', 'cpc',
                                                   'word_count', 'char_count',
                                                   'impressions_90d',
                                                   'clicks_90d',
                                                   'pageviews_90d',
                                                   'sessions_90d', 'users_90d',
                                                   'engaged_sessions_90d',
                                                   'ai_sessions_90d',
                                                   'scrol...
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['competition_level',
                                                   'content_type',
                                                   'main_intent',
                                                   'provider_used',
                                                   'model_used', 'age_tier',
                                                   'freshness_tier',
                                                   'word_count_tier',
                                                   'char_count_tier',
                                                   'impression_tier',
                                                   'position_tier',
                                                   'trend_direction'])])),
                ('classifier',
                 LogisticRegression(max_iter=1000, random_state=42))])

In [30]:
#evaluate the model
from sklearn.metrics import f1_score

y_pred = model.predict(X_test)
model_f1 = f1_score(y_test , y_pred)

print("logistic regression F1 Score:", model_f1)


logistic regression F1 Score: 0.7192771084337349


In [31]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

R_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        random_state=42,
        n_estimators=300,       # more trees = more stable, not more overfit
        max_depth=10,           # stops trees growing until leaves are pure
        min_samples_leaf=5,     # each leaf must cover ≥5 samples
        min_samples_split=10,   # each split must involve ≥10 samples
        max_features="sqrt",    # only √n features per split (standard RF default)
        class_weight="balanced" # handles class imbalance if present
    ))
])

R_model.fit(X_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['search_volume',
                                                   'competition', 'cpc',
                                                   'word_count', 'char_count',
                                                   'impressions_90d',
                                                   'clicks_90d',
                                                   'pageviews_90d',
                                                   'sessions_90d', 'users_90d',
                                                   'engaged_sessions_90d',
                                                   'ai_sessions_90d',
                                                   'scrol...
                                                  ['competition_level',
                                                   'content_type',
                                                   'main_intent',
                                                   'provider_used',
                                                   'model_used', 'age_tier',
                                                   'freshness_tier',
                                                   'word_count_tier',
                                                   'char_count_tier',
                                                   'impression_tier',
                                                   'position_tier',
                                                   'trend_direction'])])),
                ('classifier',
                 RandomForestClassifier(class_weight='balanced', max_depth=10,
                                        min_samples_leaf=5,
                                        min_samples_split=10, n_estimators=300,
                                        random_state=42))])

In [32]:
y2_pred = R_model.predict(X_test)
random_forest_metric = f1_score(y_test , y_pred)

print("Random Forest F1 Score:", random_forest_metric)

Random Forest F1 Score: 0.7192771084337349


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [33]:
error_df = test_df.copy()

error_df["actual"] = y_test.values
error_df["predicted"] = y2_pred

error_df["error"] = (
    error_df["actual"] != error_df["predicted"]
)

errors = error_df[error_df["error"]].copy()

print("Number of errors:", len(errors))

Number of errors: 2


In [34]:
false_positives = error_df[
    (error_df["actual"] == 0) &
    (error_df["predicted"] == 1)
]

false_negatives = error_df[
    (error_df["actual"] == 1) &
    (error_df["predicted"] == 0)
]

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

False positives: 1
False negatives: 1


In [35]:
# inspect the mistake
error_columns = [
    "content_id",
    "client_id",
    "avg_position",
    "content_age_days",
    "impressions_90d",
    "impressions_last_30d",
    "impressions_prev_30d",
    "ctr",
    "days_since_last_update",
    "trend_direction",
    "position_tier",
    "impression_tier",
    label,
    "predicted"
]

print(errors[error_columns])

                 content_id          client_id  avg_position  \
15586  content_80f53a9b30db  client_349c41201b           8.0   
25590  content_a22b7f6c73c5  client_7f2253d7e2           9.1   

       content_age_days  impressions_90d  impressions_last_30d  \
15586               148              248                    12   
25590               132            28192                  3149   

       impressions_prev_30d   ctr  days_since_last_update trend_direction  \
15586                    20  0.00                      20            down   
25590                  7537  4.78                      20            down   

      position_tier impression_tier is_initial_refresh_candidate  predicted  
15586        page_1             low                         True          0  
25590        page_1            good                        False          1  


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.